ResNet-50 Bottleneck Block

In [1]:
import torch
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity

torch.manual_seed(42)

In [2]:
class DepthwiseSeparableConv(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):

        super().__init__()

        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=in_channels,
            bias=False
        )

        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

    def forward(self, x):

        x = self.depthwise(x)
        x = self.pointwise(x)

        return x

In [6]:
class Bottleneck(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):

        super().__init__()

        # 1x1 Reduction
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        # Depthwise Separable Conv
        self.conv2 = DepthwiseSeparableConv(
            out_channels,
            out_channels,
            stride
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        # 1x1 Expansion
        self.conv3 = nn.Conv2d(
            out_channels,
            out_channels*4,
            kernel_size=1,
            bias=False
        )

        self.bn3 = nn.BatchNorm2d(out_channels*4)

        # Skip Projection
        self.shortcut = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels*4,
                kernel_size=1,
                stride=stride,
                bias=False
            ),
            nn.BatchNorm2d(out_channels*4)
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        identity = self.shortcut(x)

        out = self.relu(self.bn1(self.conv1(x)))

        out = self.relu(self.bn2(self.conv2(out)))

        out = self.bn3(self.conv3(out))

        out += identity

        out = self.relu(out)

        return out



In [5]:

x = torch.randn(1,64,56,56)

In [4]:

model = Bottleneck(64,64)

In [7]:

output = model(x)

print("Input Shape :",x.shape)
print("Output Shape:",output.shape)


Input Shape : torch.Size([1, 64, 56, 56])
Output Shape: torch.Size([1, 256, 56, 56])


In [8]:
total_params = sum(p.numel() for p in model.parameters())

print("\nTotal Parameters:",total_params)



Total Parameters: 42816


In [9]:
with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,
    profile_memory=True
) as prof:

    output = model(x)

print("\n========== PROFILER ==========")

print(
    prof.key_averages().table(
        sort_by="cpu_time_total",
        row_limit=10
    )
)


========== PROFILER ==========
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                aten::batch_norm         0.30%      90.302us        51.82%      15.640ms       3.910ms       7.66 MB           0 B             4  
    aten::_batch_norm_impl_index         4.63%       1.398ms        51.52%      15.549ms       3.887ms       7.66 MB           0 B             4  
         aten::native_batch_norm        46.29%      13.972ms        46.85%      14.140ms       3.535ms       7.66 MB     -10.00 KB             4  
                    aten::conv2d         0.15%      45.496us        36.94%      11.149

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
